# 1. Tratamento de Valores em Falta (NA)
Valores em falta (representados como NA - Not Available - em R) são muito comuns em datasets reais. É crucial identificá-los e decidir como lidar com eles, pois podem afetar as nossas análises.

## 1.1. Criar um Dataset de Exemplo com NAs
Vamos criar um data frame simples com alguns valores em falta para demonstração.

In [ ]:
dados_na <- data.frame(
  ID = 1:5,
  Nome = c("João", "Maria", "Pedro", "Ana", "Luís"),
  Idade = c(20, 22, NA, 21, 23),
  Pontuacao = c(85, NA, 90, 78, NA),
  Cidade = c("Lisboa", "Porto", "Coimbra", NA, "Faro")
)
print(dados_na)

## 1.2. Identificar Valores em Falta
### 1.2.1. is.na(): Verificar se é NA
A função `is.na()` retorna TRUE se um valor for NA e FALSE caso contrário.

In [ ]:
is.na(dados_na$Idade) # Verifica a coluna Idade
is.na(dados_na)       # Verifica todo o data frame

### 1.2.2. sum(is.na()): Contar NAs por Coluna
Podemos usar sum(is.na(coluna)) para contar o número de NAs numa coluna, pois TRUE é tratado como 1 e FALSE como 0.

In [ ]:
# NAs na coluna Idade
sum(is.na(dados_na$Idade))

# NAs na coluna Pontuacao
sum(is.na(dados_na$Pontuacao))

# NAs em todas as colunas (aplicando a cada coluna)
sapply(dados_na, function(x) sum(is.na(x)))

# Ou com dplyr e summarise (mais elegante para data frames)
library(dplyr)
dados_na %>%
  summarise_all(function(x) sum(is.na(x)))

### 1.2.3. complete.cases(): Identificar Linhas Completas
Esta função retorna um vetor lógico indicando quais linhas não têm NAs.

In [ ]:
complete.cases(dados_na)

# Contar o número de linhas completas
sum(complete.cases(dados_na))

## 1.3. Estratégias para Lidar com Valores em Falta
### 1.3.1. Remover Linhas com NAs (na.omit())
A forma mais simples é remover todas as linhas que contêm pelo menos um NA. Cuidado: Isto pode resultar na perda de muita informação se houver muitos NAs espalhados.

In [ ]:
dados_sem_na_omit <- na.omit(dados_na)
print(dados_sem_na_omit)

### 1.3.2. Remover Linhas com NAs (drop_na() do tidyr)
O pacote tidyr (também parte do tidyverse) tem uma função mais flexível, drop_na().

In [ ]:
# install.packages("tidyr") # Descomentem e executem se não tiverem
library(tidyr)

# Remover linhas com NA em qualquer coluna
dados_sem_na_tidyr <- dados_na %>%
  drop_na()
print(dados_sem_na_tidyr)

# Remover linhas apenas se tiverem NA em 'Idade' ou 'Pontuacao'
dados_sem_na_colunas_especificas <- dados_na %>%
  drop_na(Idade, Pontuacao)
print(dados_sem_na_colunas_especificas)

### 1.3.3. Imputação Simples (Substituir NAs)
Em vez de remover, podemos substituir os NAs por um valor. As estratégias mais comuns são:
* Média: Para variáveis numéricas.
* Mediana: Para variáveis numéricas (menos sensível a outliers).
* Moda: Para variáveis categóricas.
* Um valor fixo: Ex: 0, "Desconhecido".

**Exemplo:** Imputar NAs com a Média da Coluna

In [ ]:
# Criar uma cópia para não alterar o original
dados_imputados <- dados_na

# Calcular a média da coluna Idade (ignorando NAs)
media_idade <- mean(dados_imputados$Idade, na.rm = TRUE)
print(paste("Média da Idade:", media_idade))

# Substituir NAs na coluna Idade pela média
dados_imputados$Idade[is.na(dados_imputados$Idade)] <- media_idade
print(dados_imputados)

# Fazer o mesmo para Pontuacao
media_pontuacao <- mean(dados_imputados$Pontuacao, na.rm = TRUE)
dados_imputados$Pontuacao[is.na(dados_imputados$Pontuacao)] <- media_pontuacao
print(dados_imputados)

**Exemplo:** Imputar NAs com a Moda da Coluna (para categóricas)
Para a moda, não há uma função direta no R, mas podemos criá-la ou usar a lógica.

In [ ]:
# Função para encontrar a moda
get_mode <- function(v) {
  uniqv <- unique(v)
  uniqv[which.max(tabulate(match(v, uniqv)))]
}

# Imputar NAs na coluna Cidade com a moda
moda_cidade <- get_mode(dados_na$Cidade[!is.na(dados_na$Cidade)])
print(paste("Moda da Cidade:", moda_cidade))

dados_imputados_cidade <- dados_na
dados_imputados_cidade$Cidade[is.na(dados_imputados_cidade$Cidade)] <- moda_cidade
print(dados_imputados_cidade)

Com dplyr e tidyr para imputação:

In [ ]:
# Imputar NAs com a média usando dplyr e tidyr::replace_na
dados_imputados_dplyr <- dados_na %>%
  mutate(
    Idade = replace_na(Idade, mean(Idade, na.rm = TRUE)),
    Pontuacao = replace_na(Pontuacao, mean(Pontuacao, na.rm = TRUE))
  )
print(dados_imputados_dplyr)

**Qual estratégia usar?** Depende do contexto e da quantidade de NAs.
* **Remoção:** Simples, mas pode perder dados valiosos.
*	**Imputação:** Mantém mais dados, mas a "imputação" é uma estimativa e pode introduzir viés.

**Parabéns!** Já sabem como identificar e começar a tratar valores em falta!